# TB Portals — Frozen Backbone Comparison
Runs R1 (BMC head) on three alternative frozen CXR backbones to validate that RAD-DINO is the right choice:
1. **BioMedCLIP** (vision-language, biomedical literature)
2. **TorchXRayVision** (CXR-specific, supervised on 14 datasets)
3. **DINOv2-natural** (Facebook DINOv2 on natural images)

Same head architecture, same protocol, same 5,010-image manifest, 5 seeds, 4 modes (a1/a2/a3/fusion).
Attach only `tb-portals-cxr-pngs`. Internet **ON**. GPU T4. Estimated runtime ≈ 3 hr.

## 0 — Clone repo

In [1]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH   = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
print('repo ready at', REPO_DIR)

Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (500/500), done.


## 1 — Install deps (open_clip for BioMedCLIP)

In [2]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'open_clip_torch', 'torchxrayvision', 'pydicom',
                'pylibjpeg', 'pylibjpeg-libjpeg'], check=False)
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
deps installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

## 2 — Paths and manifest

In [3]:
import os
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
print('export exists:', os.path.isdir(KAGGLE_EXPORT))

export exists: True


In [4]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f'manifest: {len(paper_df)} images -> {PAPER_MANIFEST}')

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
manifest: 5010 images -> /kaggle/working/tbportals_manifest_paper.csv


## 3 — Cache features for each backbone (one CLS file per backbone)

In [5]:
from cache_features import main as cache_main
BACKBONES = [
    ('biomedclip',     f'{WORK}/features_biomedclip_cls.npz'),
    ('txrv',           f'{WORK}/features_txrv_cls.npz'),
    ('dinov2-natural', f'{WORK}/features_dinov2nat_cls.npz'),
]
import os
for name, path in BACKBONES:
    if os.path.isfile(path):
        print('cached ->', name, '|', path); continue
    print(f'caching {name} ...')
    cache_main(['--manifest', PAPER_MANIFEST, '--out', path,
                '--backbone', name, '--batch-size', '32'])
print('all backbones cached')

caching biomedclip ...
[cache] backbone=biomedclip device=cuda patch_grid=off (CLS)


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[cache] feature dim = 768
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cache] 2880/5010
[cache] 3200/5010
[cache] 3520/5010
[cache] 3840/5010
[cache] 4160/5010
[cache] 4480/5010
[cache] 4800/5010
[cache] wrote features (5010, 512) -> /kaggle/working/features_biomedclip_cls.npz
caching txrv ...
[cache] backbone=txrv device=cuda patch_grid=off (CLS)
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
[cache] feature dim = 1024
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cach

...

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[cache] feature dim = 768
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cache] 2880/5010
[cache] 3200/5010
[cache] 3520/5010
[cache] 3840/5010
[cache] 4160/5010
[cache] 4480/5010
[cache] 4800/5010
[cache] wrote features (5010, 768) -> /kaggle/working/features_dinov2nat_cls.npz
all backbones cached


## 4 — Train R1 (BMC) per backbone × mode × country × seed

In [6]:
from src.training.train_agentic import main as agentic_main
import os
for name, path in BACKBONES:
    for mode in ['a2', 'a3', 'fusion', 'a1']:
        outdir = f'{WORK}/baseline_{name}_{mode}'
        if os.path.isdir(outdir):
            print('skip', outdir); continue
        print(f'=== {name} | {mode} ===')
        args = ['--features', path, '--manifest', PAPER_MANIFEST,
                '--mode', mode, '--out-dir', outdir,
                '--rungs', '1', '--seeds', '0', '1', '2', '3', '4',
                '--held-outs', 'Romania', 'Moldova', 'Kazakhstan']
        # a1 needs a patch-grid cache; skip a1 for non-RAD-DINO backbones for now
        if mode == 'a1' and name != 'rad-dino':
            print('  (a1 mode needs grid cache; skipping for backbones comparison run)')
            continue
        agentic_main(args)

=== biomedclip | a2 ===
[agentic] device=cuda mode=a2 dim=512 feat=CLS grid_available=False rungs=[1] seeds=[0, 1, 2, 3, 4] M=5
[agentic] 5 configs: ['rung1_mse', 'rung1_bmc', 'agentic_best', 'agentic_best_tta', 'stacked_legacy']
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[a2|rung1_mse] Romania s0  MAE=19.37 CI[17.3,21.3] r=0.705 a=0.00 | base 20.11 paper 18.70 -> within-noise
[tbportals] split held_out=Romania: train=3836 val=954 test=220 (train/val patients 3618/904).
[a2|rung1_mse] Romania s1  MAE=20.28 CI[18.1,22.5] r=0.663 a=0.00 | base 20.11 paper 18.70 -> within-noise
[tbportals] split held_out=Romania: train=3843 val=947 test=220 (train/val patients 3618/904).
[a2|rung1_mse] Romania s2  MAE=19.09 CI[17.1,21.2] r=0.717 a=0.00 | base 20.11 paper 18.70 -> within-noise
[tbportals] split held_out=Romania: train=3843 val=947 test=220 (train/val patients 3618/904).
[a2|rung1_mse] Romania s3  MAE=19.48 CI[17.4,21.6] r=0.699 a=0.00 | b

## 5 — Compute pool-mean and kNN-only trivial baselines (CPU only)

In [7]:
import numpy as np, pandas as pd, json
from src.data.tbportals import make_country_split
from cache_features import load_features

RAD = f'{WORK}/features_rad-dino_cls.npz'  # used by kNN baseline
if not os.path.isfile(RAD):
    print('caching RAD-DINO for kNN baseline...')
    cache_main(['--manifest', PAPER_MANIFEST, '--out', RAD,
                '--backbone', 'rad-dino', '--batch-size', '32'])
feats, dim = load_features(RAD)

rows_mean, rows_knn = [], []
manifest = pd.read_csv(PAPER_MANIFEST, dtype={'image_id': str})
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    tr_df, _, te_df = make_country_split(manifest, held_out_country=country, val_fraction=0.2, seed=0)
    tr_df = tr_df.copy(); te_df = te_df.copy()
    tr_df['timika'] = tr_df['alp_0_100'] + 40 * tr_df['cavity']
    te_df['timika'] = te_df['alp_0_100'] + 40 * te_df['cavity']

    # ---- pool-mean baseline ----
    yhat = float(tr_df['timika'].mean())
    mae_mean = float(np.mean(np.abs(te_df['timika'].values - yhat)))
    rows_mean.append({'method': 'pool_mean', 'held_out': country, 'n_test': len(te_df), 'timika_mae': mae_mean})

    # ---- kNN-only baseline (K=25) ----
    X_tr = np.stack([feats[i] for i in tr_df['image_id'] if i in feats])
    y_tr = tr_df[tr_df['image_id'].isin(feats)]['timika'].values
    X_te = np.stack([feats[i] for i in te_df['image_id'] if i in feats])
    y_te = te_df[te_df['image_id'].isin(feats)]['timika'].values
    X_tr_n = X_tr / (np.linalg.norm(X_tr, axis=1, keepdims=True) + 1e-9)
    X_te_n = X_te / (np.linalg.norm(X_te, axis=1, keepdims=True) + 1e-9)
    sim = X_te_n @ X_tr_n.T
    idx = np.argsort(-sim, axis=1)[:, :25]
    yhat = y_tr[idx].mean(axis=1)
    mae_knn = float(np.mean(np.abs(y_te - yhat)))
    rows_knn.append({'method': 'knn_only', 'held_out': country, 'n_test': len(y_te), 'timika_mae': mae_knn})

    print(f'{country}: pool_mean={mae_mean:.2f} | knn_only={mae_knn:.2f}')

trivial = pd.DataFrame(rows_mean + rows_knn)
trivial.to_csv(f'{WORK}/trivial_baselines.csv', index=False)
print('saved trivial baselines')

caching RAD-DINO for kNN baseline...
[cache] backbone=rad-dino device=cuda patch_grid=off (CLS)


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[cache] feature dim = 768
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cache] 2880/5010
[cache] 3200/5010
[cache] 3520/5010
[cache] 3840/5010
[cache] 4160/5010
[cache] 4480/5010
[cache] 4800/5010
[cache] wrote features (5010, 768) -> /kaggle/working/features_rad-dino_cls.npz
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
Romania: pool_mean=29.69 | knn_only=20.67
[tbportals] split held_out=Moldova: train=3531 val=890 test=589 (train/val patients 3282/820).
Moldova: pool_mean=35.93 | knn_only=23.06
[tbportals] split held_out=Kazakhstan: train=3690 val=921 test=399 (train/val patients 3434/858).
Kazakhstan: pool_mean=33.24 | knn_only=23.21
saved trivial baselines


## 6 — Package results for download

In [8]:
import shutil, glob, os
import pandas as pd
# collect every results_agentic_*.csv across backbone runs
rows = []
for name, _ in BACKBONES:
    for mode in ['a2', 'a3', 'fusion', 'a1']:
        for f in glob.glob(f'{WORK}/baseline_{name}_{mode}/results_agentic_*.csv'):
            d = pd.read_csv(f); d['backbone'] = name; d['mode'] = mode
            rows.append(d)
if rows:
    pd.concat(rows, ignore_index=True).to_csv(f'{WORK}/results_backbones.csv', index=False)
    print('aggregated results -> results_backbones.csv')

OUT = f'{WORK}/backbones_results'
os.makedirs(OUT, exist_ok=True)
for src in [f'{WORK}/results_backbones.csv', f'{WORK}/trivial_baselines.csv']:
    if os.path.isfile(src):
        shutil.copy(src, OUT)
zip_path = shutil.make_archive(f'{WORK}/backbones_results', 'zip', OUT)
print('zip ->', zip_path)

aggregated results -> results_backbones.csv
zip -> /kaggle/working/backbones_results.zip
